In [ ]:
import

# 0) Objectif de la modélisation
L’objectif de cette section est double :
(i) proposer un modèle prédictif permettant de simuler le nombre de licenciés à partir de caractéristiques observables (dont les médailles olympiques) ;
(ii) proposer un modèle économétrique permettant d’évaluer si les médailles des derniers JO sont associées à une variation de la croissance du nombre de licenciés dans les années suivantes, toutes choses égales par ailleurs.

# 1) Jeu de données de modélisation et variables
## 1.1 Dataset final

Nous travaillons sur un panel sport–année (≈ 34 sports × années observées), obtenu en agrégeant les licences individuelles et en fusionnant les variables de médailles à la référence olympique pertinente (jo_ref).
Le dataset final df_model_full contient :

- la variable cible nb_licencies

- les médailles des JO de référence (or, argent, bronze, total_medailles)

- des variables structurelles construites à partir des licences (âge moyen, parts femmes, concentration géographique, etc.)

- des variables dynamiques (lags, croissance, rolling mean).

In [ ]:
print(df_model_full.shape)
df_model_full.head()
df_model_full.isna().mean().sort_values(ascending=False).head(10)


## 1.2 Variable cible + transformation

Les effectifs de licenciés sont très dispersés selon les sports. Pour stabiliser la variance et faciliter l’interprétation, nous utilisons une transformation logarithmique :

log_y = \log(1 + nb_licencies)
]

L’ajout de 1 évite les problèmes numériques en cas de valeur nulle.

In [ ]:
dfw = df_model_full.copy()
dfw["log_y"] = np.log1p(dfw["nb_licencies"])
dfw[["nb_licencies","log_y"]].describe()


In [ ]:
# histogramme nb lic vs logy

plt.figure()
plt.hist(dfw["nb_licencies"], bins=30)
plt.title("Distribution nb_licencies (niveau)")
plt.show()

plt.figure()
plt.hist(dfw["log_y"], bins=30)
plt.title("Distribution log(1+nb_licencies)")
plt.show()


# 2) Stratégie d’évaluation : split temporel (pas de fuite)

Pour respecter la causalité temporelle (pas de fuite d’information), l’évaluation est faite sur une période future :

train : 2017–2020

test : 2022–2024
L’année 2021 n’est pas utilisée car le nombre de licenciés n’est pas recensé.

In [ ]:
train_df = dfw[dfw["annee"].between(2017, 2020)].copy()
test_df  = dfw[dfw["annee"].between(2022, 2024)].copy()

print("Train years:", sorted(train_df["annee"].unique()), "| n =", len(train_df))
print("Test years :", sorted(test_df["annee"].unique()),  "| n =", len(test_df))


# 3) Modèle 1 — Modèle économétrique (objectif : interprétation)

Ici économétrie : coefficients + SE + p-values + discussion.

## 3.1 Intuition du modèle

Le nombre de licenciés présente une forte inertie : les effectifs d’une année dépendent fortement de l’année précédente.
Pour mesurer l’effet des médailles “toutes choses égales par ailleurs”, on estime un modèle sur la croissance log :

\Delta \log(1+L_{s,t}) = \alpha_s + \beta \cdot Medailles_{s,JO(t)} + \varepsilon_{s,t}
]

où 
𝛼
𝑠
α
s
	​

 capture les caractéristiques permanentes du sport (popularité structurelle, coûts d’accès, culture sportive…).
Les erreurs standards sont clusterisées par sport, car les observations d’un même sport sont corrélées dans le temps.

In [ ]:
import statsmodels.formula.api as smf
import numpy as np

dfe = df_model_full.copy()
dfe = dfe.sort_values(["code_sport","annee"])
dfe["log_y"] = np.log1p(dfe["nb_licencies"])
dfe["dlog_y"] = dfe.groupby("code_sport")["log_y"].diff(1)

# on garde la fenêtre où on a dlog
dfe = dfe.dropna(subset=["dlog_y", "total_medailles"])

train_e = dfe[dfe["annee"].between(2017, 2020)].copy()

ols = smf.ols("dlog_y ~ total_medailles + C(code_sport)", data=train_e).fit(
    cov_type="cluster", cov_kwds={"groups": train_e["code_sport"]}
)
print(ols.summary())


## 3.2 Comment lire β (interprétation chiffrée)

Le coefficient 
𝛽
β s’interprète comme une variation de la croissance annuelle moyenne (en log) associée à une médaille supplémentaire aux derniers JO de référence.
En approximation : une variation de 
𝛽
β correspond à environ 
100
×
𝛽
100×β % de variation.

In [ ]:
beta = ols.params["total_medailles"]
se   = ols.bse["total_medailles"]
pval = ols.pvalues["total_medailles"]
ci   = ols.conf_int().loc["total_medailles"].tolist()

print("beta =", beta)
print("se   =", se)
print("pval =", pval)
print("CI95 =", ci)
print("Effet approx (%):", 100*beta)


# 3.3 Effet par sport

RQ : un “effet causal par sport” n’est pas identifiable proprement avec juste 3 points de médailles (2016/2020/2024) et peu d’années, mais tu peux faire une estimation par sport (corrélation “within sport”) et être honnête dessus.

Les effets peuvent être hétérogènes selon les sports. Nous estimons donc une version “sport-par-sport” :

\Delta \log(1+L_t)=a + b \cdot Medailles_{JO(t)} + \varepsilon_t
]

Cette estimation est moins robuste (peu d’observations par sport) : elle doit être interprétée comme un diagnostic exploratoire.

In [ ]:
import statsmodels.api as sm

def medal_effect_for_sport(df, sport_code, medals_col="total_medailles",
                           min_year=2017, max_year=2024):
    d = df.copy()
    d = d.sort_values(["code_sport","annee"])
    d["log_y"] = np.log1p(d["nb_licencies"])
    d["dlog_y"] = d.groupby("code_sport")["log_y"].diff(1)

    ds = d[(d["code_sport"] == sport_code) & d["annee"].between(min_year, max_year)]
    ds = ds.dropna(subset=["dlog_y", medals_col])

    if len(ds) < 4:
        return {"sport": sport_code, "error": f"pas assez d'observations (n={len(ds)})"}

    X = sm.add_constant(ds[[medals_col]])
    y = ds["dlog_y"]
    m = sm.OLS(y, X).fit()

    b = float(m.params[medals_col])
    se = float(m.bse[medals_col])
    p = float(m.pvalues[medals_col])
    ci = m.conf_int().loc[medals_col].tolist()

    return {
        "sport": sport_code,
        "beta_dlog": b,
        "beta_pct_approx": 100*b,
        "se": se,
        "p_value": p,
        "ci95_dlog": tuple(ci),
        "ci95_pct_approx": (100*ci[0], 100*ci[1]),
        "n_obs": len(ds)
    }


In [ ]:
medal_effect_for_sport(df_model_full, "HAN", medals_col="total_medailles")


# 4) Modèle 2 — Modèle prédictif (objectif : simulation + graphiques)

Ici on assume qu'on veut un modèle qui prédit bien pour faire des courbes “observé vs prédit” + simulation.

## 4.1 Choix du modèle

Pour la prédiction, nous utilisons une régression Ridge sur la variable log-transformée, avec :

inertie : log(1 + nb_licencies_lag1)

tendance temporelle trend

médailles des derniers JO (or, argent, bronze ou total_medailles)

effets fixes sport (dummies sur code_sport)
La Ridge limite l’instabilité liée aux variables corrélées.

## 4.2 Code du modèle + prédictions test

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

dfp = df_model_full.copy().sort_values(["code_sport","annee"])

# features temporelles
dfp["trend"] = dfp["annee"] - dfp["annee"].min()

# target log + lag log (pas de fuite)
dfp["log_y"] = np.log1p(dfp["nb_licencies"])
dfp["log_lag1"] = dfp.groupby("code_sport")["log_y"].shift(1)

dfp = dfp.dropna(subset=["log_lag1"]).copy()

train_df = dfp[dfp["annee"].between(2017, 2020)].copy()
test_df  = dfp[dfp["annee"].between(2022, 2024)].copy()

features_num = ["log_lag1", "trend", "or", "argent", "bronze", "total_medailles"]
features_cat = ["code_sport"]

X_train = pd.get_dummies(train_df[features_cat + features_num], columns=features_cat, drop_first=True)
X_test  = pd.get_dummies(test_df[features_cat + features_num], columns=features_cat, drop_first=True)
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df["log_y"].values
y_test  = test_df["log_y"].values

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

pred_log = model.predict(X_test)
pred_nb  = np.expm1(pred_log)

results = test_df[["code_sport","annee","nb_licencies"]].copy()
results["pred_nb_licencies"] = pred_nb
results["abs_err"] = np.abs(results["nb_licencies"] - results["pred_nb_licencies"])

print("R² (log):", r2_score(y_test, pred_log))
print("MAE (niveau):", mean_absolute_error(results["nb_licencies"], results["pred_nb_licencies"]))

results.head()


## 4.3 Fonction graphique “observé vs prédit” par sport 

Afin de visualiser la qualité de la prédiction sport par sport, on trace les effectifs observés et prédits sur la période test (2022–2024).

In [ ]:
import matplotlib.pyplot as plt

def plot_pred_sport(code_sport, results_df, jo_line=2020):
    d = results_df[results_df["code_sport"] == code_sport].sort_values("annee")
    if d.empty:
        print("Sport introuvable dans results:", code_sport)
        return

    plt.figure()
    plt.plot(d["annee"], d["nb_licencies"], marker="o", label="observé")
    plt.plot(d["annee"], d["pred_nb_licencies"], marker="o", linestyle="--", label="prédit")
    plt.axvline(jo_line, linestyle="--")
    plt.text(jo_line+0.1, d["nb_licencies"].min(), f"JO {jo_line}", alpha=0.7)

    plt.title(f"{code_sport} — Nb de licenciés observé vs prédit")
    plt.xlabel("Année")
    plt.ylabel("Nb licenciés")
    plt.legend()
    plt.show()


In [ ]:
plot_pred_sport("HAN", results)


# 4.4 Résultats “par sport” (table)


Pour aller au-delà d’un indicateur global, on calcule des métriques par sport (MAE, erreur moyenne, etc.).

In [ ]:
by_sport = (results
    .groupby("code_sport")
    .agg(
        mae=("abs_err","mean"),
        mape=("abs_err", lambda x: (x / results.loc[x.index, "nb_licencies"]).mean()),
        n=("abs_err","size"),
    )
    .sort_values("mae", ascending=False)
)

by_sport.head(10)


# 5) Simulations “si X médailles aux prochains JO…”


Le modèle prédictif peut être utilisé comme outil de simulation : on modifie le nombre de médailles d’un sport (ceteris paribus) et on observe l’impact sur les prédictions 2022–2024.
Attention : cela reste un exercice de simulation conditionnelle, pas une preuve causale.

In [ ]:
def simulate_medals(code_sport, delta_total_medals, df_test, X_train_cols, fitted_model):
    sim = df_test.copy()
    mask = sim["code_sport"] == code_sport
    sim.loc[mask, "total_medailles"] = np.maximum(0, sim.loc[mask, "total_medailles"] + delta_total_medals)

    X_sim = pd.get_dummies(sim[features_cat + features_num], columns=features_cat, drop_first=True)
    X_sim = X_sim.reindex(columns=X_train_cols, fill_value=0)

    pred_log_sim = fitted_model.predict(X_sim)
    pred_nb_sim  = np.expm1(pred_log_sim)

    out = sim[["code_sport","annee","nb_licencies"]].copy()
    out["pred_nb_sim"] = pred_nb_sim
    return out

# exemple : +3 médailles total pour HAN
sim_han = simulate_medals("HAN", 3, test_df, X_train.columns, model)
sim_han[sim_han["code_sport"]=="HAN"]


In [ ]:
def plot_simulation(code_sport, base_results, sim_out):
    b = base_results[base_results["code_sport"]==code_sport].sort_values("annee")
    s = sim_out[sim_out["code_sport"]==code_sport].sort_values("annee")

    plt.figure()
    plt.plot(b["annee"], b["nb_licencies"], marker="o", label="observé")
    plt.plot(b["annee"], b["pred_nb_licencies"], marker="o", linestyle="--", label="prédit (base)")
    plt.plot(s["annee"], s["pred_nb_sim"], marker="o", linestyle="--", label="prédit (simulation)")
    plt.title(f"{code_sport} — Simulation médailles")
    plt.xlabel("Année")
    plt.ylabel("Nb licenciés")
    plt.legend()
    plt.show()

plot_simulation("HAN", results, sim_han)


# 6) Discussion : pourquoi les prédictions sont souvent en dessous ?


On observe que, pour certains sports, le modèle sous-prédit systématiquement. Plusieurs raisons plausibles :

Ruptures structurelles non observées : engouement médiatique, politiques fédérales, effet Paris 2024, etc.

Variables omises : couverture TV, financement, infrastructures locales…

Modèle log + Ridge : la régularisation tend à “ramener vers la moyenne” et peut réduire les pics.

Peu d’années et peu de JO : l’effet “médailles → licences” est difficile à identifier statistiquement avec seulement 2016/2020/2024.

# 7) Limites et forces

## Forces

split temporel (reproductible, pas de fuite)

construction de features riche à partir des micro-données (âge, sexe, géographie)

double approche : interprétable (OLS) + performante (Ridge)

## Limites

faible nombre d’années disponibles → puissance statistique limitée

identification causale difficile : les médailles ne sont pas exogènes (sports déjà forts)

résultats économétriques à interpréter comme association conditionnelle, sauf hypothèses fortes